In [12]:
import idx2numpy # type: ignore
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import random
import torchvision.transforms.functional as TF

In [ ]:
# (60000, 28, 28) images, (60000,) labels
images_2d = idx2numpy.convert_from_file("../data/train-images.idx3-ubyte")  # type: ignore
images = images_2d.reshape(images_2d.shape[0], -1)  # type: ignore
images_scaled = images / 255
labels = idx2numpy.convert_from_file("../data/train-labels.idx1-ubyte")  # type: ignore

test_images_2d = idx2numpy.convert_from_file("../data/t10k-images.idx3-ubyte")  # type: ignore
test_images = test_images_2d.reshape(test_images_2d.shape[0], -1)  # type: ignore
test_images_scaled = test_images / 255
test_labels = idx2numpy.convert_from_file("../data/t10k-labels.idx1-ubyte")  # type: ignore

In [48]:
X_train = torch.tensor(images_scaled, dtype=torch.float32)
Y_train = torch.tensor(labels, dtype=torch.long)
X_test = torch.tensor(test_images_scaled, dtype=torch.float32)
Y_test = torch.tensor(test_labels, dtype=torch.long)

train_dataset = TensorDataset(X_train, Y_train)
test_dataset = TensorDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [49]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.main(x)

In [50]:
model = MyModel()
learning_rate = 0.05
beta = 0.9
epochs = 45
rotation_epoch = 15

optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=beta)
criterion = nn.CrossEntropyLoss()

In [51]:
for epoch in range(epochs):
    model.train()  # Enable Dropout and gradients
    use_rotation = epoch < rotation_epoch

    for batch_X, batch_Y in train_loader:
        if use_rotation:
            angle = random.uniform(-15, 15)
            tx, ty = random.uniform(-2, 2), random.uniform(-2, 2)

            # Affine logic (Batch processing)
            batch_X_2d = batch_X.view(-1, 1, 28, 28)
            rotated_2d = TF.affine(
                batch_X_2d,
                angle=angle,
                translate=[tx, ty],
                scale=1.0,
                shear=0.0,
                interpolation=TF.InterpolationMode.BILINEAR,
            )
            batch_X = rotated_2d.view(-1, 784)

        # Optimization Step
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)
        loss.backward()
        optimizer.step()

    # Manual LR Decay (Matches your NumPy logic)
    for param_group in optimizer.param_groups:
        param_group["lr"] *= 0.96

In [52]:
model.eval()  # Disable Dropout
correct = 0
with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        outputs = model(batch_X)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == batch_Y).sum().item()

print(f"Final Test Accuracy: {(correct / len(Y_test)) * 100:.2f}%")

Final Test Accuracy: 98.92%


In the end we have an extra dropout layer which makes all the neurons more generalized
Tried to make every image be randomly transformed by took multiple minutes so batches of image transforms is prob the play